# 📐 ERCE 2019 · Cuba — Modelos y estimación (Módulo Nacional de Matemática)

**3er grado · objetivos derivados de `CUBA3_MAT_MODULO_NACIONAL.csv`**

Este cuaderno ejecuta el proceso completo de construcción y estimación de
modelos sobre los tres datasets que deja `exploracion.ipynb`:

| Tarea | Tipo | Objetivo |
|---|---|---|
| `reg_indice_global` | regresión | `Indice_global` (0–100) |
| `clf_indice_bajo` | clasificación binaria | `bajo` = 0–40 vs 60–100 |
| `clf_indice_nivel` | clasificación multiclase | banda `bajo` / `medio` / `alto` |

No se llama a `modelos/experimento.py`: **el proceso está reescrito aquí paso a
paso**, usando los mismos componentes del paquete (`preprocesamiento`,
`seleccion_atributos`, `balanceo`, `metricas`, `friedman`, `sobreajuste`,
`plots`). Así el lector ve la decisión, no solo el resultado.

> **Lo que se espera encontrar.** En `exploracion.ipynb` §7 se mostró que las
> tres fuentes del módulo (estudiante, docente, directivo) son ortogonales entre
> sí, y que ninguna variable disponible se asocia con el objetivo
> (|r| máximo = 0,06). Con ese punto de partida, lo esperable es que los modelos
> **no superen la línea base** de predecir siempre lo más frecuente. El cuaderno
> está construido para reportar ese resultado con la misma stringentía que si
> fuera un modelo bueno: cada número se muestra junto a su línea base.

## 0. Configuración

Solo infraestructura: semilla, rutas de artefactos y silenciado de warnings
benignos (convergencia de SMO/MLP, varianza nula en una clase, pivotes del
discretizador). Las versiones se imprimen para que el resultado sea
reproducible.

In [ ]:
import json
import os
import sys
import time
import warnings

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (KFold, RandomizedSearchCV, RepeatedKFold,
                                     RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score)
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from statsmodels.stats.multitest import multipletests

# La raíz del proyecto es el directorio donde vive este cuaderno (o su padre).
RAIZ = os.path.abspath(".")
for _ in range(3):
    if os.path.isdir(os.path.join(RAIZ, "modelos")):
        break
    RAIZ = os.path.dirname(RAIZ)
sys.path.insert(0, RAIZ)

from modelos import algoritmos_extendidos as EXT
from modelos import balanceo as BAL
from modelos import config as mcfg
from modelos import contrato as CTR
from modelos import friedman as FRI
from modelos import metricas as MET
from modelos import plots as PLT
from modelos import sobreajuste as SAJ
from modelos.clasificadores import obten_clasificadores
from modelos.preprocesamiento import construir_preprocesador, nombres_transformados
from modelos.regresores import obten_regresores
from modelos.seleccion_atributos import ETIQUETAS_WEEKA, crear_selector

SEMILLA = mcfg.SEED
# Artefactos propios de este objetivo: no se tocan los de los 5 modelos del bloque de modelos.
ARTEFACTOS = os.path.join(RAIZ, "artefactos", "indice_global")
REPORTE = os.path.join(ARTEFACTOS, "reporte")
os.makedirs(REPORTE, exist_ok=True)

PLT.configurar_matplotlib()
for _c in (ConvergenceWarning, RuntimeWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_c)

print(f"raíz          : {RAIZ}")
print(f"semilla       : {SEMILLA}")
print(f"artefactos    : {os.path.relpath(ARTEFACTOS, RAIZ)}/")
print(f"pandas {pd.__version__} · numpy {np.__version__} · "
      f"sklearn {__import__('sklearn').__version__} · imblearn {__import__('imblearn').__version__}")

## 1. Qué se modela

Los tres datasets los escribe `exploracion.ipynb` en `modelos/datos/` con la
convención del proyecto: **las 28 variables primero y el objetivo en la última
columna**. Aquí se cargan, se verifica el contrato y se calcula el índice de
desbalance (IR) que gobierna la regla de balanceo:

| IR | Acción |
|---|---|
| ≤ 1,2 | sin balancear |
| 1,2 – 5 | SMOTE |
| 5 – 9 | SMOTE + submuestreo |
| > 9 | SMOTEENN |

El IR se calcula **por fold**, no sobre todo el dataset: es una decisión de
entrenamiento y no puede ver la distribución del fold de validación.

In [ ]:
TAREAS = {
    "reg_indice_global": {
        "tipo": "regresion", "objetivo": "Indice_global",
        "descripcion": "Índice global de percepción del estudiante (0–100)"},
    "clf_indice_bajo": {
        "tipo": "clasificacion", "objetivo": "bajo",
        "descripcion": "Percepción baja (0–40) vs alta (60–100)"},
    "clf_indice_nivel": {
        "tipo": "clasificacion", "objetivo": "nivel",
        "descripcion": "Banda de percepción: bajo / medio / alto"},
}
CATEGORICAS = ["provincia", "género", "area_esc"]

DATAFRAMES = {}
filas = []
for _tarea, _spec in TAREAS.items():
    _ruta = os.path.join(mcfg.DATOS_DIR, f"{_tarea}.csv")
    assert os.path.exists(_ruta), f"falta {_ruta}: ejecuta antes exploracion.ipynb"
    _df = pd.read_csv(_ruta)
    DATAFRAMES[_tarea] = _df
    _y = _df[_spec["objetivo"]]
    _ir = BAL.calcular_ir(_y.to_numpy()) if _spec["tipo"] == "clasificacion" else np.nan
    _est = (BAL.elegir_balanceo(_ir, multiclase=_y.nunique() > 2)
            if _spec["tipo"] == "clasificacion" else "—")
    filas.append({
        "tarea": _tarea, "tipo": _spec["tipo"], "n": f"{len(_df):,}".replace(",", "."),
        "objetivo": _spec["objetivo"],
        "distribución": (f"{_y.mean():.1f} ± {_y.std():.1f}"
                         if _spec["tipo"] == "regresion"
                         else {str(k): int(v) for k, v in _y.value_counts().items()}),
        "IR": "—" if np.isnan(_ir) else round(_ir, 2),
        "balanceo por regla": "no aplica" if np.isnan(_ir) else (_est or "sin balancear"),
    })

resumen_datos = pd.DataFrame(filas)
resumen_datos

In [ ]:
# El preprocesador recibe listas explícitas: `género` y `area_esc` son códigos
# numéricos (0/1) pero deben tratarse como categóricas, y las columnas nuevas
# (edad_anos, log_num_hab_esc) no están en `cfg.FEATURES_NUMERICOS`.
for _tarea, _df in DATAFRAMES.items():
    _vars = [c for c in _df.columns if c != TAREAS[_tarea]["objetivo"]]
    _num = [c for c in _vars if c not in CATEGORICAS]
    assert len(_num) + len(CATEGORICAS) == len(_vars), _tarea
    assert set(CATEGORICAS) <= set(_vars), _tarea
print(f"espacio de variables: {len(_num)} numéricas + {len(CATEGORICAS)} categóricas = {len(_vars)}")
print("categóricas:", CATEGORICAS)
print("numéricas :", _num)

# Contexto: qué variables sobrevivieron a la selección de exploracion.ipynb §11
_sel = pd.read_csv(os.path.join(mcfg.DATOS_DIR, "seleccion_indice_global.csv"))
_top = (_sel.groupby("variable", as_index=False)
        .agg(folds=("folds", "sum"), evaluadores=("evaluadores", "mean"))
        .sort_values(["folds", "evaluadores"], ascending=False))
_top.head(12)

## 2. El proceso, paso a paso

Es el flujo de `docs/metodologia.md`, que es a su vez el del
`Experimenter` de Weka. Cada fila de esta tabla es una función de este cuaderno.

| # | Paso | Función | Qué garantiza |
|---|---|---|---|
| 1 | Hold-out 20 % | `paso_1_holdout` | reserva que **nunca** participa en la selección |
| 2 | Grid de configuraciones | `paso_2_grid` | dos barridos: 4 selectores × familias baratas, y CFS × familias caras |
| 3 | CV 5 folds | `paso_3_cv_seleccion` | preprocesador y selector se ajustan **dentro** del fold |
| 3b | Afinado del top-5 | `paso_3b_afinar` | `RandomizedSearchCV` con el sampler dentro del pipeline |
| 3c | Contra la línea base | `paso_3c_pareado` | t pareado + Wilcoxon + Holm, fold a fold |
| 4 | Comparativa | `paso_4_comparativa` | media ± DE por configuración |
| 5 | Friedman + Nemenyi | `paso_5_friedman` | ¿las diferencias entre algoritmos son significativas? |
| 6 | Cadena ganadora | `paso_6_cadena_final` | pre → selección → (balanceo) → modelo |
| 7 | Hold-out imparcial | `paso_7_holdout` | métricas **contra la línea base** |
| 8 | CV 10 folds | `paso_8_cv_final` | default de Weka, cadena completa por fold |
| 9 | Estabilidad 3×5 | `paso_9_estabilidad` | ¿el número del paso 7 aguanta? |
| 10 | Sobreajuste | `paso_10_sobreajuste` | brecha train–test y curva de aprendizaje |
| 11 | Importancia | `paso_11_importancias` | qué variables usa el ganador |
| 12 | Persistencia | `paso_12_persistencia` | pipeline de despliegue + metadata |

Sobre la rejilla del paso 2: el registro ampliado
(`modelos/algoritmos_extendidos.py`) **no toca** el registro base de Weka que
sustenta los cinco modelos de `artefactos/`. Solo estas tres tareas usan el
ampliado. Añadir familias sin más habría dejado los cinco modelos sin
reproducir, porque `experimento.py` recorre el registro completo en cada
entrenamiento.

## 3. Paso 1 — Hold-out reservado (20 %)

El 20 % se aparta **antes** de mirar nada. No se usa para seleccionar selector,
algoritmo ni balanceo: solo para la estimación final. Todo lo demás ocurre en el
80 % restante.

Para clasificación el corte es estratificado, de modo que la reserva conserve la
proporción de clases.

In [ ]:
def principal_de(tipo):
    # Métrica principal de cada tipo, como en `experimento._principal`.
    return "f1_macro" if tipo == "clasificacion" else "rmse"


def es_min(tipo):
    # En regresión, RMSE: menor es mejor.
    return tipo == "regresion"


def scoring_de(tipo):
    # sklearn exige `greater_is_better` coherente con la métrica.
    return "f1_macro" if tipo == "clasificacion" else "neg_root_mean_squared_error"


def probas(modelo, X, clases=None):
    # Probabilidades para clasificadores, alineadas a `clases` y renormalizadas.
    #
    # El fallback a `decision_function` devuelve puntuaciones, no probabilidades
    # (en multiclase no suman 1), y `roc_auc_score` las rechaza. Además, un
    # modelo ajustado en un fold donde una clase quedó ausente devuelve fewer
    # columnas que clases hay. Se corrige aquí para que la métrica de cada
    # algoritmo sea comparable con la de los demás.
    P = None
    if hasattr(modelo, "predict_proba"):
        try:
            P = np.asarray(modelo.predict_proba(X), dtype=float)
        except Exception:
            P = None
    if P is None and hasattr(modelo, "decision_function"):
        d = np.asarray(modelo.decision_function(X), dtype=float)
        if d.ndim == 1:
            P = np.c_[1 - d, d]
        elif d.ndim == 2 and d.shape[1] == 2:
            P = np.c_[1 - d[:, 1], d[:, 1]]
        else:
            e = np.exp(d - d.max(axis=1, keepdims=True))
            P = e / e.sum(axis=1, keepdims=True)
    if P is None or P.ndim != 2:
        return None
    propias = list(getattr(modelo, "classes_", range(P.shape[1])))
    if clases is not None and list(propias) != list(clases):
        alineada = np.zeros((P.shape[0], len(clases)))
        for j, c in enumerate(propias):
            if c in clases:
                alineada[:, clases.index(c)] = P[:, j]
        P = alineada
    suma = P.sum(axis=1, keepdims=True)
    P = np.divide(P, suma, out=np.full_like(P, 1.0 / P.shape[1]),
                  where=suma > 0)
    return P


def nombre_original(nombre):
    # `provincia_La_Habana` -> `provincia` (one-hot a variable original).
    for c in CATEGORICAS:
        if nombre.startswith(c + "_"):
            return c
    return nombre


def paso_1_holdout(tarea):
    df = DATAFRAMES[tarea]
    spec = TAREAS[tarea]
    X = df.drop(columns=[spec["objetivo"]])
    y = df[spec["objetivo"]]
    Xtr, Xte, ytr, yte = SAJ.dividir_holdout(X, y, spec["tipo"])
    return Xtr, Xte, ytr, yte


for _tarea in TAREAS:
    _Xtr, _Xte, _ytr, _yte = paso_1_holdout(_tarea)
    _tipo = TAREAS[_tarea]["tipo"]
    _dist = (f"{_ytr.mean():.1f} ± {_ytr.std():.1f}" if _tipo == "regresion"
             else {str(k): int(v) for k, v in _ytr.value_counts().items()})
    _dist_te = (f"{_yte.mean():.1f} ± {_yte.std():.1f}" if _tipo == "regresion"
                else {str(k): int(v) for k, v in _yte.value_counts().items()})
    print(f"{_tarea:20s} selección {len(_ytr):>4d} filas  |  hold-out {len(_yte):>4d} filas")
    print(f"{'':20s}   selección: {_dist}")
    print(f"{'':20s}   hold-out : {_dist_te}")

## 4. Paso 2 — El grid de configuraciones

Se cruzan cuatro evaluadores de atributos (los de Weka que ya usa el proyecto)
con los algoritmos de cada tipo y con las dos opciones de balanceo: la que pide
la regla del IR y `none`. **El modelo sin balancear siempre entra**: si
balancear no mejora la métrica, se conserva el modelo sin balancear, igual que en
Weka.

| Tipo | Algoritmos |
|---|---|
| clasificación | NaiveBayes, IBk, J48, SMO, Logistic, RandomForest, MultilayerPerceptron |
| regresión | LinearRegression (ridge), SMOreg, IBk, M5P, RandomForest, MultilayerPerceptron |

In [ ]:
# Registro ampliado: sustituye al base (`obten_clasificadores` / `obten_regresores`)
# solo para estas tres tareas. Los cinco modelos de `artefactos/` siguen
# reproduciéndose con el registro base, que este módulo no toca.
EXTENDIDOS = {"clasificacion": EXT.obten_clasificadores_extendidos,
              "regresion": EXT.obten_regresores_extendidos}

N_AFINADO_TOP = 5


def fabricas_de(tarea):
    # Registro de factories de la tarea (el mismo en los dos barridos).
    return EXTENDIDOS[TAREAS[tarea]["tipo"]](SEMILLA)


def modelo_afinado(tarea, config):
    # Instancia el modelo ganador con sus hiperparámetros afinados. El
    # afinado se hizo dentro de un pipeline, así que sus nombres vienen
    # prefijados con `mod__`; aquí se quita ese prefijo y se aplican al
    # estimador sin envolver. Los que ya venían anidados (por ejemplo
    # `estimator__C` del SMO calibrado) se dejan como están.
    modelo = fabricas_de(tarea)[config["algoritmo"]]()
    params = {k[len("mod__"):] if k.startswith("mod__") else k: v
              for k, v in (config.get("parametros") or {}).items()}
    if params:
        modelo.set_params(**params)
    return modelo


def paso_2_grid(tarea):
    # Los dos barridos del cribado: (etiqueta, selectores, algoritmos, fabricas)
    #
    # Barrido A: los cuatro selectores de Weka × las familias baratas.
    # Barrido B: solo CfsSubsetEval (el selector por defecto de Weka) × las
    # familias caras y los ensembles. Multiplicar por cuatro selectores el coste
    # de un ensemble de tres modelos no cambia el ranking de algoritmos.
    tipo = TAREAS[tarea]["tipo"]
    fabricas = EXTENDIDOS[tipo](SEMILLA)
    nivel_a, nivel_b = EXT.grupos(tipo)
    return [("A", list(mcfg.SELECTORES_NOMBRES), nivel_a, fabricas),
            ("B", ["cfs"], nivel_b, fabricas)]


def resumen_grid():
    # Cuántas configuraciones cuesta cada tarea, por barrido.
    lineas = []
    for _tarea in TAREAS:
        tipo = TAREAS[_tarea]["tipo"]
        _ytr = paso_1_holdout(_tarea)[2]
        _ir = (BAL.calcular_ir(np.asarray(_ytr)) if tipo == "clasificacion"
               else None)
        _estr = (BAL.elegir_balanceo(_ir, multiclase=len(set(_ytr)) > 2)
                 if _ir else None)
        _bal = ["none"] + ([_estr] if _estr else [])
        for _et, _sels, _mods, _ in paso_2_grid(_tarea):
            n = len(_sels) * len(_mods) * len(_bal) * mcfg.CV_FOLDS_SELECCION
            lineas.append({"tarea": _tarea, "barrido": _et, "tipo": tipo,
                           "selectores": len(_sels), "algoritmos": len(_mods),
                           "balanceos": len(_bal),
                           "algoritmos_nombres": ", ".join(_mods),
                           "configuraciones": n})
    return pd.DataFrame(lineas)


GRID = resumen_grid()
for _r in GRID.itertuples():
    print(f"{_r.tarea:20s} barrido {_r.barrido} · {_r.tipo[:5]:5s} · "
          f"{_r.selectores} selectores × {_r.algoritmos} algoritmos × "
          f"{_r.balanceos} balanceos × {mcfg.CV_FOLDS_SELECCION} folds = "
          f"{_r.configuraciones:5d} evaluaciones")
print(f"\n  total: {GRID['configuraciones'].sum()} evaluaciones de modelo")

print()
for _n, _et in ETIQUETAS_WEEKA.items():
    print(f"  {_n:9s} → {_et}")

print("\n  algoritmos del cribado (nivel A con los 4 selectores, nivel B con CFS):")
for _tipo in ("clasificacion", "regresion"):
    _a, _b = EXT.grupos(_tipo)
    print(f"    {_tipo:14s} A({len(_a)}): {', '.join(_a)}")
    print(f"    {'':14s} B({len(_b)}): {', '.join(_b)}")

## 5. Paso 3 — Validación cruzada de selección (5 folds)

El detalle que hace que esto no sea una fuga: dentro de cada fold se ajusta
**el preprocesador** (imputación, escalado, one-hot) y **el selector** sobre las
solo filas de entrenamiento, y luego se transforman las de validación con esos
mismos objetos ajustados.

Los tres evaluadores que no dependen del modelo (CFS, χ²/F, InfoGain) se ajustan
una vez por fold y se reutilizan para los 7 algoritmos; el `WrapperSubsetEval`
depende del estimador base, así que se reentrena por algoritmo, igual que en
`experimento.py`.

In [ ]:
def paso_3_cv_seleccion(tarea, Xtr, ytr, verbose=True):
    # Devuelve una fila por (fold, selector, algoritmo, balanceo) con las métricas.
    spec = TAREAS[tarea]
    tipo = spec["tipo"]
    clasif = tipo == "clasificacion"
    principal = principal_de(tipo)
    numericos = [c for c in Xtr.columns if c not in CATEGORICAS]
    clases = [str(c) for c in pd.Series(ytr).dropna().unique()] if clasif else None
    barridos = paso_2_grid(tarea)

    folds = (StratifiedKFold(mcfg.CV_FOLDS_SELECCION, shuffle=True, random_state=SEMILLA)
             if clasif else
             KFold(mcfg.CV_FOLDS_SELECCION, shuffle=True, random_state=SEMILLA))

    filas = []
    omitidos = []
    for fold_n, (tr, va) in enumerate(folds.split(Xtr, ytr)):
        t0 = time.time()
        # --- preprocesador: ajustado SOLO con el fold de entrenamiento
        pre = construir_preprocesador(numericos=numericos, categoricos=CATEGORICAS)
        X_pp = pre.fit_transform(Xtr.iloc[tr])
        X_vp = pre.transform(Xtr.iloc[va])
        y_tr = np.asarray(ytr.iloc[tr])
        y_va = np.asarray(ytr.iloc[va])

        # --- balanceo: decidido con el IR de este fold
        ir_fold = BAL.calcular_ir(y_tr) if clasif else None
        estrategia = (BAL.elegir_balanceo(ir_fold, multiclase=len(clases) > 2)
                      if clasif else None)
        balanceos = ["none"] + ([estrategia] if estrategia else [])

        for etiqueta, selectores_n, modelos_n, fabricas in barridos:
            n_config = len(selectores_n) * len(modelos_n) * len(balanceos)
            # --- evaluadores que no dependen del modelo (se ajustan una vez)
            ajustados = {}
            for sel_n in selectores_n:
                if sel_n == "wrapper":
                    continue
                s = crear_selector(sel_n, tipo, seed=SEMILLA)
                s.fit(X_pp, y_tr)
                ajustados[sel_n] = (s.transform(X_pp), s.transform(X_vp))

            for sel_n in selectores_n:
                for mod_n in modelos_n:
                    if sel_n == "wrapper" and mod_n in EXT.SIN_WRAPPER:
                        omitidos.append((etiqueta, sel_n, mod_n))
                        continue
                    if sel_n == "wrapper":
                        s = crear_selector("wrapper", tipo,
                                           estimador=fabricas[mod_n](), seed=SEMILLA)
                        s.fit(X_pp, y_tr)
                        Xs_tr, Xs_va = s.transform(X_pp), s.transform(X_vp)
                    else:
                        Xs_tr, Xs_va = ajustados[sel_n]
                    for bal in balanceos:
                        # El balanceo se aplica solo al entrenamiento, nunca a la validación.
                        Xb, yb = BAL.aplicar_balanceo(Xs_tr, y_tr, bal, seed=SEMILLA)
                        modelo = fabricas[mod_n]()
                        modelo.fit(Xb, yb)
                        yp = modelo.predict(Xs_va)
                        mf = (MET.metricas_fold_clasif(y_va, yp, probas(modelo, Xs_va, clases), clases)
                              if clasif else MET.metricas_fold_regresion(y_va, yp))
                        filas.append({"fold": fold_n, "barrido": etiqueta,
                                      "selector": sel_n,
                                      "selector_weka": ETIQUETAS_WEEKA[sel_n],
                                      "algoritmo": mod_n, "balanceo": bal,
                                      "ir_fold": ir_fold, **mf})
            if verbose:
                print(f"    fold {fold_n + 1}/{mcfg.CV_FOLDS_SELECCION} · barrido "
                      f"{etiqueta}: {n_config:3d} configuraciones · "
                      f"{time.time() - t0:5.1f} s")

    tabla = pd.DataFrame(filas)
    if omitidos:
        unicos = sorted({f"{m} × {sl}" for _, sl, m in omitidos})
        tabla.attrs["omitidos"] = unicos
        if verbose:
            print(f"  · omitido el wrapper sobre {len(unicos)} combinaciones "
                  f"(ensembles): {', '.join(unicos)}")
    if verbose:
        print(f"  → {len(tabla)} evaluaciones · "
              f"{len(barridos)} barridos · {len(balanceos)} balanceos × "
              f"{mcfg.CV_FOLDS_SELECCION} folds")
    return tabla


# Vista previa del formato de la tabla: un solo fold de la tarea binaria.
_Xtr, _Xte, _ytr, _yte = paso_1_holdout("clf_indice_bajo")
_tr, _va = next(iter(StratifiedKFold(mcfg.CV_FOLDS_SELECCION, shuffle=True,
                                     random_state=SEMILLA).split(_Xtr, _ytr)))
_pre = construir_preprocesador(numericos=[c for c in _Xtr.columns if c not in CATEGORICAS],
                              categoricos=CATEGORICAS)
_pp = _pre.fit_transform(_Xtr.iloc[_tr])
_s = crear_selector("infogain", "clasificacion", seed=SEMILLA).fit(_pp, np.asarray(_ytr.iloc[_tr]))
print("columnas tras preprocesar :", _pp.shape[1])
print("variables que selecciona InfoGainRanker (top-8 de Weka):")
print([n for i, n in enumerate(nombres_transformados(_pre)) if _s.support_[i]])

## 6. Paso 3b — Afinar el top-5 (búsqueda de hiperparámetros)

El cribado anterior usa los valores por defecto de cada algoritmo (los mismos
de Weka). Aquí los mejores **(selector, algoritmo, balanceo)** refinan sus
hiperparámetros con `RandomizedSearchCV`: 20 iteraciones sobre 5 folds con la
misma métrica principal.

Dos decisiones que importan para que el número sea comparable:

- **El balanceo va dentro del pipeline** (`imblearn.Pipeline`), no aplicado antes
  de la búsqueda. Así cada fold de la CV interna se equilibra por separado, que
  es lo que hace el proceso de Weka; si se equilibrara una sola vez fuera, los
  folds de validación quedarían contaminados con datos sintéticos.
- **El afinado nunca toca el hold-out**: usa solo la parte de selección
  (`Xtr`, `ytr`). El hold-out se reserva para la evaluación del paso 7.

Con `WrapperSubsetEval` la búsqueda se reduce a 10 iteraciones: el selector ya
es una búsqueda exhaustiva sobre subconjuntos y cada evaluación del wrapper es
una CV del modelo completo.

| Algoritmo | Qué se busca |
|---|---|

In [ ]:
# Qué se afina en cada familia, leído del módulo (no escrito a mano).
pd.DataFrame([{"algoritmo": _n,
               "parametros": ", ".join(_d),
               "combinaciones": int(np.prod([len(v) for v in _d.values()]))}
              for _t in ("clasificacion", "regresion")
              for _n, _d in EXT.ESPACIOS_AJUSTE[_t].items()])

Los ensembles quedan fuera a propósito: afinar un `Voting` o un `Stacking`
sin haber afinado sus componentes solo produce sobreajuste. El módulo lo expone
en `ESPACIOS_AJUSTE`, y `EXT.afinables()` es el filtro que usa el orquestador.

In [ ]:
def paso_3b_afinar(tarea, Xtr, ytr, top, verbose=True):
    # Refina los `top` con RandomizedSearchCV, sin tocar el hold-out. `top` son
    # filas de la comparativa del cribado, ya ordenadas. Devuelve (resumen,
    # detalle) con la mejor configuración de cada algoritmo y todas las
    # iteraciones evaluadas: hay que poder leer qué se buscó, no solo qué se ganó.
    tipo = TAREAS[tarea]["tipo"]
    fabricas = EXTENDIDOS[tipo](SEMILLA)
    numericos = [c for c in Xtr.columns if c not in CATEGORICAS]
    y = np.asarray(ytr)
    filas, detalle = [], []

    for cfg in top:
        mod_n, sel_n, bal = cfg["algoritmo"], cfg["selector"], cfg["balanceo"]
        dist = EXT.espacio_ajuste(mod_n, tipo)
        if dist is None:
            continue
        n_iter = 10 if sel_n == "wrapper" else EXT.N_AFINADO
        t0 = time.time()

        # El preprocesador y el selector se ajustan con todo Xtr: esto es la CV
        # interna, el hold-out sigue intacto.
        pre = construir_preprocesador(numericos=numericos, categoricos=CATEGORICAS)
        X_pp = pre.fit_transform(Xtr)
        s = crear_selector(sel_n, tipo, estimador=fabricas[mod_n](), seed=SEMILLA)
        s.fit(X_pp, y)
        Xs = s.transform(X_pp)
        n_sel = Xs.shape[1]

        # El sampler entra en el pipeline: cada fold se equilibra por separado.
        pasos = ([("bal", BAL.crear_balanceador(bal, seed=SEMILLA))] if bal != "none"
                 else []) + [("mod", fabricas[mod_n]())]
        # Los nombres del espacio llevan el prefijo del paso del pipeline.
        dist_pipe = {f"mod__{k}": v for k, v in dist.items()}
        busca = RandomizedSearchCV(ImbPipeline(pasos), dist_pipe, n_iter=n_iter,
                                   cv=EXT.CV_AFINADO, scoring=scoring_de(tipo),
                                   random_state=SEMILLA, n_jobs=-1, refit=True)
        busca.fit(Xs, y)

        # `RandomizedSearchCV` puntúa con la métrica de sklearn (RMSE negado);
        # aquí se pasa a la escala de la tarea para que el delta sea legible.
        base_score = float(cfg.get("principal_mean", np.nan))
        cv_afinado = (float(busca.best_score_) if tipo == "clasificacion"
                      else float(-busca.best_score_))
        filas.append({"algoritmo": mod_n, "selector": sel_n,
                      "selector_weka": ETIQUETAS_WEEKA[sel_n], "balanceo": bal,
                      "n_variables": n_sel, "iteraciones": n_iter,
                      "cv_por_defecto": base_score,
                      "cv_afinado": cv_afinado,
                      "delta": cv_afinado - base_score,
                      "segundos": round(time.time() - t0, 1),
                      "parametros": busca.best_params_})
        signo = 1.0 if tipo == "clasificacion" else -1.0
        for k in range(len(busca.cv_results_["params"])):
            detalle.append({"algoritmo": mod_n, "selector": sel_n, "iteracion": k,
                            "cv_mean": signo * float(busca.cv_results_["mean_test_score"][k]),
                            "cv_std": float(busca.cv_results_["std_test_score"][k]),
                            "parametros": str(busca.cv_results_["params"][k])})
        if verbose:
            print(f"    {mod_n:22s} {sel_n:9s} bal={bal:5s} "
                  f"defecto={base_score:8.4f} → afinado={busca.best_score_:8.4f} "
                  f"({filas[-1]['delta']:+.4f}) en {filas[-1]['segundos']} s")

    if not filas:
        raise ValueError("ningún algoritmo del top tiene espacio de búsqueda")
    resumen = (pd.DataFrame(filas)
               .sort_values("cv_afinado", ascending=es_min(tipo))
               .reset_index(drop=True))
    return resumen, pd.DataFrame(detalle)

## 7. Paso 3c — ¿Alguien supera a la línea base de verdad?

Con `|r| ≤ 0,06` en estas variables, la pregunta importante no es qué algoritmo
gana el cribado, sino si **cualquier** algoritmo supera lo que se consigue
adivinando siempre lo más frecuente. Con 16 algoritmos por tarea son 48
comparaciones: alguna gana por azar.

La respuesta exige mirar los mismos folds, no el promedio:

1. Se recalcula la **línea base fold a fold** (clase mayoritaria o media del
   objetivo) con los mismos cortes que el cribado.
2. Para cada algoritmo se toma su **mejor configuración en cada fold**.
3. Se calcula la diferencia contra la base en cada fold y se contrasta con un
   **t pareado** y con **Wilcoxon**.
4. Como son muchas comparaciones, se corrige con **Holm**: decide el mejor `p`
   ajustado, no el mejor `p` crudo.

Con 5 folds la potencia del contraste es baja, así que la tabla también lleva la
diferencia media y en cuántos folds gana cada algoritmo. Uno que mejora la base
en 1 de 5 folds no ha demostrado nada.

In [ ]:
def paso_3c_pareado(tabla, tarea, Xtr, ytr):
    # Compara cada algoritmo con la línea base fold a fold (t pareado + Holm).
    tipo = TAREAS[tarea]["tipo"]
    clasif = tipo == "clasificacion"
    principal = principal_de(tipo)
    y = pd.Series(np.asarray(ytr))
    clases = [str(c) for c in y.dropna().unique()] if clasif else None
    folds = (StratifiedKFold(mcfg.CV_FOLDS_SELECCION, shuffle=True, random_state=SEMILLA)
             if clasif else
             KFold(mcfg.CV_FOLDS_SELECCION, shuffle=True, random_state=SEMILLA))

    # 1) línea base en cada uno de los mismos folds
    base = {}
    for fold_n, (tr, va) in enumerate(folds.split(Xtr, y)):
        if clasif:
            # Predecir siempre la clase mayoritaria: el AUC queda en 0,5 y el
            # F1 es el de la constante, que es la referencia que hay que superar.
            mayoritario = str(y.iloc[tr].mode().iloc[0])
            pred = np.array([mayoritario] * len(va))
            P = np.zeros((len(va), len(clases)))
            P[:, clases.index(mayoritario)] = 1.0
            base[fold_n] = MET.metricas_fold_clasif(y.iloc[va], pred, P, clases)[principal]
        else:
            base[fold_n] = MET.metricas_fold_regresion(
                y.iloc[va], np.full(len(va), float(y.iloc[tr].mean())))[principal]
    base = pd.Series(base)

    # 2) mejor configuración de cada algoritmo en cada fold
    _m = tabla[principal] if clasif else -tabla[principal]
    idx = tabla.assign(_m=_m).groupby(["fold", "algoritmo"])["_m"].idxmax()
    mejor = tabla.loc[idx]
    filas = []
    for mod_n, gr in mejor.groupby("algoritmo"):
        g = gr.set_index("fold").reindex(base.index)
        m_alg = (g[principal] if clasif else -g[principal]).astype(float)
        d = (m_alg - base).dropna()
        n = len(d)
        if n > 1 and float(d.std(ddof=1)) > 0:
            t_p = float(stats.ttest_rel(m_alg, base).pvalue)
            try:
                w_p = float(stats.wilcoxon(d.values, zero_method="wilcox").pvalue)
            except ValueError:
                w_p = float("nan")
        else:
            t_p = w_p = float("nan")
        filas.append({"algoritmo": mod_n, "folds": n,
                      "metrica_media": float(m_alg.mean()),
                      "base_media": float(base.mean()),
                      "diferencia": float(d.mean()),
                      "de_diferencia": float(d.std(ddof=1)) if n > 1 else np.nan,
                      "folds_gana": int((d > 0).sum()),
                      "t_pareado_p": t_p, "wilcoxon_p": w_p})

    res = pd.DataFrame(filas)
    # 4) corrección de Holm sobre toda la familia de algoritmos
    res["p_ajustado_holm"] = multipletests(res["t_pareado_p"].fillna(1.0),
                                            method="holm")[1]
    res["supera_la_base"] = np.where(
        res["p_ajustado_holm"] < 0.05,
        np.where(res["diferencia"] > 0, "sí", "peor que la base"),
        "no se puede afirmar")
    res = res.sort_values("diferencia", ascending=not clasif).reset_index(drop=True)
    res.attrs["linea_base_por_fold"] = base
    return res

## 8. Paso 4 — Comparativa de configuraciones

Se agrega por configuración (media ± DE sobre los 5 folds) y se ordena por la
métrica principal. **El mismo fold aporta a todas las configuraciones**, que es
lo que permite luego el test de Friedman: la comparación es pareada.

In [ ]:
def paso_4_comparativa(tabla, tarea):
    tipo = TAREAS[tarea]["tipo"]
    principal = principal_de(tipo)
    cols = (["exactitud", "f1_macro", "auc_macro", "kappa"] if tipo == "clasificacion"
            else ["rmse", "mae", "r2"])
    comp = tabla.groupby(["barrido", "selector", "selector_weka",
                          "algoritmo", "balanceo"])[cols].agg(["mean", "std"])
    comp.columns = [f"{m}_{s}" for m, s in comp.columns]
    comp = comp.reset_index()
    comp["principal_mean"] = comp[f"{principal}_mean"].astype(float)
    comp["principal_de"] = comp[f"{principal}_std"].astype(float)
    return comp.sort_values("principal_mean", ascending=not es_min(tipo)).reset_index(drop=True)

## 9. Paso 5 — Friedman + Nemenyi

Weka no compara algoritmos con la media: los compara con el **test de Friedman**
usando el fold como bloque, y si hay diferencias, el post-hoc de Nemenyi con su
diagrama de diferencia crítica.

Dos decisiones explícitas:

1. Para cada fold y algoritmo se toma **la mejor configuración de ese
   algoritmo**, no la media de todas: así el test responde "¿el mejor J48 es mejor
   que el mejor SMO?".
2. `friedman.rangos_medios` ordena por valor **descendente** (rango 1 = valor
   más alto). En regresión la métrica es el RMSE y menos es mejor, así que se le
   pasa el RMSE **negado**: rango 1 = mejor. Sin esta corrección el diagrama
   ordenaría al revés.

In [ ]:
def paso_5_friedman(tabla, tarea):
    tipo = TAREAS[tarea]["tipo"]
    principal = principal_de(tipo)
    metrica = tabla[principal] if tipo == "clasificacion" else -tabla[principal]
    mejor_por_fold = pd.DataFrame({"fold": tabla["fold"],
                                   "algoritmo": tabla["algoritmo"],
                                   "metrica": metrica})
    # La unidad de comparación es la mejor configuración de cada fold, no el
    # promedio de los cuatro selectores: el promedio premiaría al algoritmo
    # que acierta en un selector y falla en los otros tres.
    mejor_por_fold = mejor_por_fold.loc[
        mejor_por_fold.groupby(["fold", "algoritmo"])["metrica"].idxmax()].reset_index(drop=True)
    if mejor_por_fold["algoritmo"].nunique() < 3:
        raise ValueError("Friedman necesita al menos 3 algoritmos")
    fr = FRI.ranking_final(FRI.tabla_por_algoritmo(mejor_por_fold, "algoritmo"))
    fr["mejor_por_fold"] = mejor_por_fold
    return fr

## 10. Paso 6 — La cadena ganadora

La configuración ganadora se entrena sobre **todo el set de selección** (el 80 %)
con la cadena completa y en este orden:

```
preprocesar → seleccionar atributos → [balancear si el IR lo pide] → modelo
```

El balanceo **no** queda dentro del pipeline que se persiste: se aplica al
entrenar y se documenta en `metadata.json`, igual que en el bloque de modelos,
para que el software de escritorio solo necesite scikit-learn + joblib.

In [ ]:
def paso_6_cadena_final(tarea, Xtr, ytr, config):
    # Entrena la cadena ganadora sobre el set de selección y devuelve el pipeline.
    tipo = TAREAS[tarea]["tipo"]
    numericos = [c for c in Xtr.columns if c not in CATEGORICAS]
    fabricas = fabricas_de(tarea)

    modelo = modelo_afinado(tarea, config)
    pre = construir_preprocesador(numericos=numericos, categoricos=CATEGORICAS)
    X_pp = pre.fit_transform(Xtr)

    # El wrapper evalúa subconjuntos con el modelo, así que necesita los
    # hiperparámetros afinados: seleccionar con los valores por defecto y
    # luego predecir con otros sería medir dos modelos distintos.
    selector = crear_selector(config["selector"], tipo,
                              estimador=modelo_afinado(tarea, config),
                              seed=SEMILLA)
    selector.fit(X_pp, np.asarray(ytr))
    X_sel = selector.transform(X_pp)

    estrategia = (BAL.elegir_balanceo(BAL.calcular_ir(np.asarray(ytr)),
                                     multiclase=len(np.unique(ytr)) > 2)
                  if tipo == "clasificacion" else None)
    if config["balanceo"] != "none" and estrategia:
        X_fit, y_fit = BAL.aplicar_balanceo(X_sel, np.asarray(ytr), estrategia, seed=SEMILLA)
    else:
        X_fit, y_fit = X_sel, np.asarray(ytr)
    modelo.fit(X_fit, y_fit)

    nombres_pp = nombres_transformados(pre)
    nombres_sel = [n for i, n in enumerate(nombres_pp) if selector.support_[i]]
    pipe = Pipeline([("pre", pre), ("sel", selector), ("mod", modelo)])
    info = {"nombres_pp": nombres_pp, "nombres_sel": nombres_sel,
            "estrategia_aplicada": estrategia,
            "filas_entrenamiento": int(len(y_fit))}
    return pipe, info

## 11. Paso 7 — Evaluación imparcial sobre el hold-out

Este es el número que se reporta. Y se reporta **junto a la línea base**: predecir
siempre lo más frecuente (clasificación) o la media (regresión).

| Tipo | Referencia |
|---|---|
| clasificación | clase mayoritaria: exactitud ≈ 0,51, F1-macro ≈ 0,35, AUC 0,5, kappa 0 |
| regresión | media del objetivo: R² = 0, RMSE = DE del objetivo |

Un modelo que no supera su línea base no es un modelo: es ruido con muchos
números alrededor. Por eso `detectar_sobreajuste` comprueba **primero** si el
modelo supera la base, y solo después mira la brecha.

In [ ]:
def paso_7_holdout(tarea, pipe, Xtr, ytr, Xte, yte):
    tipo = TAREAS[tarea]["tipo"]
    clasif = tipo == "clasificacion"
    pre, sel, mod = pipe.named_steps["pre"], pipe.named_steps["sel"], pipe.named_steps["mod"]
    clases = [str(c) for c in pd.Series(yte).dropna().unique()] if clasif else None

    Xte_s = sel.transform(pre.transform(Xte))
    y_te_pred = mod.predict(Xte_s)
    y_te_prob = probas(mod, Xte_s, clases) if clasif else None
    Xtr_s = sel.transform(pre.transform(Xtr))
    y_tr_pred = mod.predict(Xtr_s)

    if clasif:
        base = MET.linea_base_clasificacion(yte)
        mm = MET.calcular_metricas_clasificacion(yte, y_te_pred, y_te_prob, clases)
        hold = {k: mm[k] for k in ("exactitud", "f1_macro", "auc_macro", "kappa")}
        base_h = {k: base[k] for k in ("exactitud", "f1_macro", "auc_macro", "kappa")}
        mt = MET.metricas_fold_clasif(ytr, y_tr_pred,
                                  probas(mod, Xtr_s, clases), clases)["f1_macro"]
        supera = hold["f1_macro"] - base_h["f1_macro"]
        brecha = SAJ.detectar_sobreajuste(mt, hold["f1_macro"], linea_base=base_h["f1_macro"])
        etiqueta = "f1_macro"
    else:
        base = MET.linea_base_regresion(yte)
        mm = MET.calcular_metricas_regresion(yte, y_te_pred)
        hold = {k: mm[k] for k in ("r", "mae", "rmse", "rae", "rrse", "r2")}
        base_h = {k: base[k] for k in ("r", "mae", "rmse", "r2")}
        mt = MET.metricas_fold_regresion(ytr, y_tr_pred)["rmse"]
        # La métrica va negada (mayor = mejor) y el piso de 0.45 no aplica a un RMSE:
        # aquí el suelo es la media del objetivo, también negada.
        supera = hold["r2"] - base_h["r2"]
        brecha = SAJ.detectar_sobreajuste(-mt, -hold["rmse"], piso=None,
                                          linea_base=-base_h["rmse"])
        etiqueta = "r2"
    base_h["supera_a_linea_base"] = round(float(supera), 4)

    return {
        "holdout": hold, "linea_base_holdout": base_h, "detalle": mm,
        "y_te_pred": y_te_pred, "y_te_prob": y_te_prob, "clases": clases,
        "brecha": brecha, "metrica_principal": etiqueta,
        "metrica_entrenamiento": round(float(mt), 4),
    }

## 12. Paso 8 — CV de 10 folds (default de Weka)

La configuración ganadora se vuelve a entrenar **completa** en cada uno de los
10 folds del set de selección: preprocesador, selector, balanceo y modelo. No se
reutiliza el modelo ya ajustado.

Es el número que se compara con el hold-out: si el 10-fold da 0,42 y el hold-out
0,58, la diferencia es ruido de muestreo, no desempeño.

In [ ]:
def paso_8_cv_final(tarea, Xtr, ytr, config, k=mcfg.CV_FOLDS_FINAL):
    tipo = TAREAS[tarea]["tipo"]
    clasif = tipo == "clasificacion"
    folds = (StratifiedKFold(k, shuffle=True, random_state=SEMILLA) if clasif
             else KFold(k, shuffle=True, random_state=SEMILLA))
    clases = [str(c) for c in pd.Series(ytr).dropna().unique()] if clasif else None
    lista = []
    for tr, va in folds.split(Xtr, ytr):
        pipe_f, _ = paso_6_cadena_final(tarea, Xtr.iloc[tr], ytr.iloc[tr], config)
        pre, sel, mod = (pipe_f.named_steps["pre"], pipe_f.named_steps["sel"],
                         pipe_f.named_steps["mod"])
        Xs = sel.transform(pre.transform(Xtr.iloc[va]))
        yp = mod.predict(Xs)
        lista.append(MET.metricas_fold_clasif(ytr.iloc[va], yp, probas(mod, Xs, clases), clases)
                     if clasif else MET.metricas_fold_regresion(ytr.iloc[va], yp))
    dfr = pd.DataFrame(lista)
    return {"detalle": dfr,
            **{f"{c}_media": round(float(dfr[c].mean()), 4) for c in dfr.columns},
            **{f"{c}_de": round(float(dfr[c].std()), 4) for c in dfr.columns}}

## 13. Paso 9 — Estabilidad (3 repeticiones × 5 folds)

Tres repeticiones de 5 folds con semillas distintas sobre el mismo 80 %. Si la
media ± DE se solapa con el valor del hold-out, el número del paso 7 es
estable; si no, era una muestra afortunada.

In [ ]:
def paso_9_estabilidad(tarea, Xtr, ytr, config):
    tipo = TAREAS[tarea]["tipo"]
    clasif = tipo == "clasificacion"
    numericos = [c for c in Xtr.columns if c not in CATEGORICAS]
    fabricas = fabricas_de(tarea)
    modelo = modelo_afinado(tarea, config)
    # El pipeline incluye el selector (se reentrena por fold) pero no el balanceo,
    # igual que el pipeline de despliegue.
    pipe = Pipeline([
        ("pre", construir_preprocesador(numericos=numericos, categoricos=CATEGORICAS)),
        ("sel", crear_selector(config["selector"], tipo, estimador=modelo, seed=SEMILLA)),
        ("mod", modelo),
    ])
    rk = (RepeatedStratifiedKFold(n_splits=5, n_repeats=mcfg.REPETICIONES_VALIDACION,
                                  random_state=SEMILLA) if clasif
          else RepeatedKFold(n_splits=5, n_repeats=mcfg.REPETICIONES_VALIDACION,
                             random_state=SEMILLA))
    scores = cross_val_score(pipe, Xtr, ytr, cv=rk, scoring=scoring_de(tipo), n_jobs=1)
    return {"scores": scores,
            "media": round(float(scores.mean()), 4), "de": round(float(scores.std()), 4),
            "reps": int(len(scores)), "metrica": scoring_de(tipo)}

## 14. Paso 10 — Control de sobreajuste

Dos herramientas, complementarias:

- **Brecha train–test** sobre el set de selección y el hold-out, con el
  diagnóstico de `sobreajuste.detectar_sobreajuste`: `nulo` (no supera la base),
  `sobreajuste`, `underfitting` u `ok`.
- **Curva de aprendizaje**: si el puntaje de validación sigue subiendo al
  aumentar los datos, el cuello de botella es la muestra, no el modelo.

La curva se calcula con la cadena completa (pre → selección → modelo) a cinco
tamaños entre el 20 % y el 100 % del set de selección.

In [ ]:
def paso_10_sobreajuste(tarea, Xtr, ytr, config):
    tipo = TAREAS[tarea]["tipo"]
    numericos = [c for c in Xtr.columns if c not in CATEGORICAS]
    fabricas = fabricas_de(tarea)
    modelo = modelo_afinado(tarea, config)
    pipe = Pipeline([
        ("pre", construir_preprocesador(numericos=numericos, categoricos=CATEGORICAS)),
        ("sel", crear_selector(config["selector"], tipo, estimador=modelo, seed=SEMILLA)),
        ("mod", modelo),
    ])
    curva = SAJ.curva_aprendizaje(pipe, Xtr, ytr, tipo, scoring_de(tipo),
                                  cv=mcfg.CV_FOLDS_SELECCION,
                                  sizes=np.linspace(0.2, 1.0, 5))
    return curva, {"resumen": SAJ.resumen_curva(curva),
                  "gancho_validacion": round(curva[-1]["test_media"] - curva[0]["test_media"], 4)}

## 15. Paso 11 — Importancia de variables

Orden de preferencia: `feature_importances_` (árboles) → `coef_` (lineales, en
valor absoluto) → importancia por permutación sobre el hold-out, que es la única
que funciona para modelos sin ningún atributo de importancia nativo. La
importancia por permutación se calcula sobre una muestra de 1.500 filas del
hold-out.

In [ ]:
def paso_11_importancias(tarea, pipe, Xte, yte, nombres_sel):
    tipo = TAREAS[tarea]["tipo"]
    pre, sel, mod = pipe.named_steps["pre"], pipe.named_steps["sel"], pipe.named_steps["mod"]
    Xte_s = sel.transform(pre.transform(Xte))
    n = min(mcfg.MUESTRA_PERMUTACION, Xte_s.shape[0])

    if hasattr(mod, "feature_importances_"):
        imp = np.abs(np.asarray(mod.feature_importances_, dtype=float))
        metodo = f"feature_importances_ ({type(mod).__name__})"
    elif hasattr(mod, "coef_"):
        c = np.asarray(mod.coef_, dtype=float)
        imp = np.abs(c.mean(axis=0) if c.ndim > 1 else c)
        metodo = f"|coef_| ({type(mod).__name__})"
    else:
        rng = np.random.RandomState(SEMILLA)
        idx = rng.choice(Xte_s.shape[0], n, replace=False)
        pi = permutation_importance(mod, Xte_s[idx], np.asarray(yte)[idx],
                                    scoring=scoring_de(tipo), n_repeats=5,
                                    random_state=SEMILLA, n_jobs=-1)
        imp = pi.importances_mean
        metodo = "permutación (5 repeticiones)"

    df = pd.DataFrame({"variable": [nombre_original(n) for n in nombres_sel],
                       "columna_preprocesada": list(nombres_sel),
                       "importancia": np.round(imp, 5)})
    # Las columnas one-hot de una misma variable se agregan: importa leer
    # "provincia", no sus 24 niveles repetidos.
    df = (df.groupby("variable", as_index=False)
            .agg(importancia=("importancia", "sum"),
                 niveles=("columna_preprocesada", "count")))
    return df.sort_values("importancia", ascending=False).reset_index(drop=True), metodo

## 16. Figuras del reporte

Las mismas figuras que genera `modelos/plots.py` (PNG 300 dpi + SVG + PDF), pero
escritas en `artefactos/indice_global/reporte/<tarea>/` para no mezclarse con los
cinco modelos del bloque de modelos.

In [ ]:
def guardar_figuras(tarea, res, pipe, comp, tabla, fr, curva, importancias, ytr):
    rep = os.path.join(REPORTE, tarea)
    os.makedirs(rep, exist_ok=True)
    tipo = TAREAS[tarea]["tipo"]
    principal = principal_de(tipo)

    # Una barra por algoritmo: su mejor configuración.
    mejor_por_algo = comp.loc[comp.groupby("algoritmo")["principal_mean"].apply(
        lambda s: s.idxmin() if es_min(tipo) else s.idxmax(), include_groups=False)]
    PLT.grafico_comparativa(mejor_por_algo.rename(columns={"principal_mean": principal}),
                            principal, rep, f"comparativa_{tarea}",
                            f"Comparativa · {tarea} ({principal})")

    if tabla["balanceo"].nunique() > 1 and tipo == "clasificacion":
        PLT.grafico_comparativa_balanceo(tabla, rep, f"balanceo_{tarea}")

    PLT.guardar_fig(FRI.grafico_cd(fr["ranking"], fr["cd_nemenyi"]), rep,
                    f"friedman_cd_{tarea}")

    detalle = res["detalle"]
    if tipo == "clasificacion":
        PLT.grafico_matriz_confusion(res["yte"], res["y_te_pred"], res["clases"], rep,
                                     f"matriz_confusion_{tarea}",
                                     f"Matriz de confusión · {tarea}")
        if res["y_te_prob"] is not None:
            PLT.grafico_roc(res["yte"], res["y_te_prob"], res["clases"], rep, f"roc_{tarea}",
                            f"Curvas ROC · {tarea}")
        PLT.grafico_ir(ytr, rep, f"clases_{tarea}", f"Distribución de la clase · {tarea}")
    else:
        PLT.grafico_scatter_regresion(res["yte"], res["y_te_pred"], rep, f"scatter_{tarea}",
                                      f"Observado vs predicho · {tarea}",
                                      r2=res["holdout"]["r2"])
        PLT.grafico_residuos(res["yte"], res["y_te_pred"], rep, f"residuos_{tarea}",
                             f"Residuos · {tarea}")

    PLT.grafico_curva_aprendizaje(curva, rep, f"aprendizaje_{tarea}",
                                  f"Curva de aprendizaje · {tarea}")
    PLT.grafico_importancia(importancias, rep, f"importancia_{tarea}",
                            f"Importancia de atributos · {tarea}")

    # Correlación entre los atributos que el ganador seleccionó.
    nombres_pp = res["nombres_pp"]
    idx = [i for i in range(len(nombres_pp)) if pipe.named_steps["sel"].support_[i]]
    if len(idx) > 2:
        idx = idx[::max(1, len(idx) // 10)]
        Xc = pipe.named_steps["pre"].transform(res["Xtr"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            corr = np.corrcoef(Xc[:, idx].T)
        PLT.grafico_correlaciones([nombres_pp[i] for i in idx], corr, rep,
                                  f"correlaciones_{tarea}",
                                  f"Correlación entre atributos seleccionados · {tarea}")
    return sorted(os.listdir(rep))

## 17. Paso 12 — Persistencia

Por tarea se escribe en `artefactos/indice_global/<tarea>/`:

- `pipeline.joblib`: pipeline de despliegue (`pre` → `sel` → `mod`) **entrenado
  con todos los datos**, con el `metadata` incrustado para que la app lo cargue
  en una sola lectura.
- `metadata.json`: contrato de entrada (variables esperadas), configuración
  ganadora, y toda la estimación (hold-out, línea base, CV 10 folds, estabilidad,
  brecha, curva de aprendizaje) y el resultado de Friedman.

Y en `artefactos/indice_global/reporte/<tarea>/` las tablas y figuras.

In [ ]:
def paso_12_persistencia(tarea, res, pipe, comp, tabla, fr, curva, importancias):
    spec = TAREAS[tarea]
    tipo = spec["tipo"]
    X = DATAFRAMES[tarea].drop(columns=[spec["objetivo"]])
    y = DATAFRAMES[tarea][spec["objetivo"]]
    config = res["config"]
    carpeta = os.path.join(ARTEFACTOS, tarea)
    rep = os.path.join(REPORTE, tarea)
    os.makedirs(carpeta, exist_ok=True)
    os.makedirs(rep, exist_ok=True)

    artefacto = {
        "modelo_id": tarea, "tarea": tarea, "tipo": tipo,
        "descripcion": spec["descripcion"], "target": spec["objetivo"],
        "clases": res["clases"],
        "features_esperadas": [str(c) for c in X.columns],
        "features_por_preproceso": res["nombres_pp"],
        "features_seleccionadas": res["nombres_sel"],
        "algoritmo": config["algoritmo"], "selector": config["selector"],
        "selector_weka": ETIQUETAS_WEEKA[config["selector"]],
        "balanceo": config["balanceo"],
        "estrategia_balanceo_ir": res["estrategia_aplicada"],
        "balanceo_despliegue": config["balanceo"],
        "importancia_metodo": res["metodo_importancia"],
        "estimacion": {
            "holdout": res["holdout"],
            "linea_base_holdout": res["linea_base_holdout"],
            "cv_weka": {k: v for k, v in res["cv_final"].items() if k != "detalle"},
            "estabilidad_3x5cv": {k: v for k, v in res["estabilidad"].items() if k != "scores"},
            "brecha": {k: res["brecha"][k] for k in ("estado", "detalle", "brecha")},
            "curva_aprendizaje": [{"n": c["n_entrenamiento"], "train": c["train_media"],
                                   "test": c["test_media"]} for c in curva],
            "aprendizaje": res["aprendizaje"],
        },
        "friedman": {"stat": fr["friedman_stat"], "p": fr["friedman_p"],
                     "significativo": fr["significativo"],
                     "cd": round(fr["cd_nemenyi"], 4),
                     "ranking": fr["ranking"].to_dict("records")},
        "afinado": {"parametros": config.get("parametros", {}),
                    "cv_por_defecto": float(res["afinado"].iloc[0]["cv_por_defecto"]),
                    "cv_afinado": float(res["afinado"].iloc[0]["cv_afinado"]),
                    "iteraciones": EXT.N_AFINADO, "folds": EXT.CV_AFINADO,
                    "top": len(res["afinado"])},
        "contra_la_linea_base": {
            "metodo": "t pareado + Wilcoxon sobre los mismos folds, corrección de Holm",
            "algoritmos_superan": int((res["pareado"]["supera_la_base"] == "sí").sum()),
            "algoritmos_probados": int(len(res["pareado"])),
            "mejor_diferencia": float(res["pareado"].iloc[0]["diferencia"]),
            "mejor_algoritmo": res["pareado"].iloc[0]["algoritmo"],
            "detalle": res["pareado"].to_dict("records")},
        "fecha": pd.Timestamp.now().isoformat(),
        "librerias": {"sklearn": __import__("sklearn").__version__,
                      "numpy": np.__version__, "pandas": pd.__version__},
    }

    # Tablas del reporte
    tabla.to_csv(os.path.join(rep, "fold_detalle.csv"), index=False)
    comp.to_csv(os.path.join(rep, "comparativa.csv"), index=False)
    fr["ranking"].to_csv(os.path.join(rep, "friedman_ranking.csv"), index=False)
    fr["nemenyi"].to_csv(os.path.join(rep, "friedman_nemenyi.csv"))
    res["cv_final"]["detalle"].to_csv(os.path.join(rep, "cv10_detalle.csv"), index=False)
    res["afinado"].to_csv(os.path.join(rep, "afinado.csv"), index=False)
    res["afinado_detalle"].to_csv(os.path.join(rep, "afinado_iteraciones.csv"), index=False)
    res["pareado"].to_csv(os.path.join(rep, "contra_la_linea_base.csv"), index=False)

    # Tablas del artefacto
    importancias.to_csv(os.path.join(carpeta, "importancia.csv"), index=False)
    pd.Series(res["holdout"]).to_frame("valor").to_csv(
        os.path.join(carpeta, "metricas_holdout.csv"))
    if tipo == "clasificacion":
        res["detalle"]["por_clase"].to_csv(os.path.join(carpeta, "por_clase.csv"), index=False)
        pd.DataFrame(res["detalle"]["matriz"], columns=res["detalle"]["clases"],
                     index=res["detalle"]["clases"]).to_csv(
            os.path.join(carpeta, "confusion.csv"))

    with open(os.path.join(carpeta, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(artefacto, f, ensure_ascii=False, indent=2, default=str)

    # Pipeline de despliegue: TODOS los datos, sin balanceo dentro.
    fabricas = fabricas_de(tarea)
    modelo = modelo_afinado(tarea, config)
    pre = construir_preprocesador(numericos=[c for c in X.columns if c not in CATEGORICAS],
                                  categoricos=CATEGORICAS)
    X_pp = pre.fit_transform(X)
    sel = crear_selector(config["selector"], tipo, estimador=modelo_afinado(tarea, config),
                         seed=SEMILLA)
    sel.fit(X_pp, np.asarray(y))
    Xs = sel.transform(X_pp)
    # El despliegue respeta la configuración ganadora: si el cribado eligió no
    # balancear, el modelo final tampoco se balancea. Balancear aquí daría un
    # pipeline distinto del que se midió en el hold-out.
    est = (BAL.elegir_balanceo(BAL.calcular_ir(np.asarray(y)), multiclase=len(np.unique(y)) > 2)
           if (tipo == "clasificacion" and config["balanceo"] != "none") else None)
    Xf, yf = (BAL.aplicar_balanceo(Xs, np.asarray(y), est, seed=SEMILLA) if est
              else (Xs, np.asarray(y)))
    modelo.fit(Xf, yf)
    joblib.dump({"pipeline": Pipeline([("pre", pre), ("sel", sel), ("mod", modelo)]),
                 "metadata": artefacto},
                os.path.join(carpeta, "pipeline.joblib"), compress=3)
    return artefacto

## 18. Orquestación: las tres tareas completas

Una función que encadena los doce pasos, en el mismo orden que
`modelos/experimento.py`. El detalle está en las celdas anteriores; aquí solo se
muestran las marcas de tiempo y el resultado de cada tarea.

In [ ]:
def ejecutar_tarea(tarea, verbose=True):
    t_ini = time.time()
    spec = TAREAS[tarea]
    tipo = spec["tipo"]
    clasif = tipo == "clasificacion"
    principal = principal_de(tipo)
    _log = print if verbose else (lambda *a, **k: None)

    # --- paso 1
    Xtr, Xte, ytr, yte = paso_1_holdout(tarea)
    _log(f"\n=== {tarea} · {spec['descripcion']} · n={len(DATAFRAMES[tarea])}")

    # --- paso 3 (el grid se imprime aparte, en §4)
    tabla = paso_3_cv_seleccion(tarea, Xtr, ytr, verbose=verbose)

    # --- paso 4
    comp = paso_4_comparativa(tabla, tarea)
    _log(f"  mejor configuración por defecto: {comp.iloc[0]['algoritmo']} "
         f"({comp.iloc[0]['selector']}, balanceo={comp.iloc[0]['balanceo']}) → "
         f"{principal} = {comp.iloc[0]['principal_mean']:.4f}")

    # --- paso 3c: contraste contra la línea base, fold a fold
    pareado = paso_3c_pareado(tabla, tarea, Xtr, ytr)
    _ganadores = pareado[pareado["supera_la_base"] == "sí"]
    _log(f"  contra la línea base (t pareado + Holm): "
         f"{len(_ganadores)} de {len(pareado)} algoritmos se pueden dar por "
         f"superadores; mejor diferencia {pareado.iloc[0]['diferencia']:+.4f} "
         f"({pareado.iloc[0]['algoritmo']})")

    # --- paso 3b: afinado del top-5. El ganador definitivo sale de aquí.
    top = comp.head(N_AFINADO_TOP).to_dict("records")
    _log(f"  afinando el top-{len(top)} con RandomizedSearchCV "
         f"({EXT.N_AFINADO} iteraciones, {EXT.CV_AFINADO} folds):")
    afinado, afinado_detalle = paso_3b_afinar(tarea, Xtr, ytr, top, verbose=verbose)
    mejor = afinado.iloc[0]
    config = {"selector": mejor["selector"], "algoritmo": mejor["algoritmo"],
              "balanceo": mejor["balanceo"],
              "parametros": dict(mejor["parametros"])}
    _log(f"  ganador tras afinar: {config['algoritmo']} ({config['selector']}, "
         f"balanceo={config['balanceo']}) afinado → {principal} = "
         f"{mejor['cv_afinado']:.4f} (por defecto {mejor['cv_por_defecto']:.4f})")

    # --- paso 5
    fr = paso_5_friedman(tabla, tarea)
    _log(f"  Friedman: p={fr['friedman_p']:.4f} "
         f"({'significativo' if fr['significativo'] else 'no significativo'}), "
         f"CD={fr['cd_nemenyi']:.2f}")

    # --- paso 6 (con los hiperparámetros afinados)
    pipe, info = paso_6_cadena_final(tarea, Xtr, ytr, config)
    _log(f"  cadena: {len(info['nombres_pp'])} columnas tras preprocesar → "
         f"{len(info['nombres_sel'])} seleccionadas: {info['nombres_sel']}")

    # --- paso 7
    res_h = paso_7_holdout(tarea, pipe, Xtr, ytr, Xte, yte)
    etq = res_h["metrica_principal"]
    _log(f"  hold-out: {etq}={res_h['holdout'][etq]:.4f} vs línea base "
         f"{res_h['linea_base_holdout'][etq]:.4f} → "
         f"{'supera' if res_h['linea_base_holdout']['supera_a_linea_base'] > 0 else 'NO supera'} "
         f"por {abs(res_h['linea_base_holdout']['supera_a_linea_base']):.4f}")
    _log(f"  brecha: {res_h['brecha']['estado']} ({res_h['brecha']['detalle']})")

    # --- pasos 8 y 9
    cv_final = paso_8_cv_final(tarea, Xtr, ytr, config)
    _log(f"  CV {mcfg.CV_FOLDS_FINAL} folds: {principal}_media={cv_final[f'{principal}_media']:.4f} "
         f"± {cv_final[f'{principal}_de']:.4f}")
    estab = paso_9_estabilidad(tarea, Xtr, ytr, config)
    _log(f"  estabilidad {mcfg.REPETICIONES_VALIDACION}×5: {estab['media']:.4f} "
         f"± {estab['de']:.4f} ({estab['metrica']})")

    # --- paso 10
    curva, aprendizaje = paso_10_sobreajuste(tarea, Xtr, ytr, config)
    _log(f"  curva de aprendizaje: validación {curva[0]['test_media']:.3f} → "
         f"{curva[-1]['test_media']:.3f}")

    # --- paso 11
    importancias, metodo = paso_11_importancias(tarea, pipe, Xte, yte, info["nombres_sel"])
    _log(f"  importancia ({metodo}): "
         f"{', '.join(importancias.variable.head(3))}")

    res = {"config": config, **res_h, "tabla": tabla, "comparativa": comp,
           "afinado": afinado, "afinado_detalle": afinado_detalle,
           "pareado": pareado,
           "friedman": fr, "cv_final": cv_final, "estabilidad": estab,
           "curva": curva, "aprendizaje": aprendizaje,
           "importancias": importancias, "metodo_importancia": metodo,
           "nombres_pp": info["nombres_pp"], "nombres_sel": info["nombres_sel"],
           "estrategia_aplicada": info["estrategia_aplicada"],
           "Xtr": Xtr, "ytr": ytr, "Xte": Xte, "yte": yte,
           "segundos": round(time.time() - t_ini, 1)}

    # --- figuras y persistencia
    res["figuras"] = guardar_figuras(tarea, res, pipe, comp, tabla, fr, curva,
                                     importancias, ytr)
    res["artefacto"] = paso_12_persistencia(tarea, res, pipe, comp, tabla, fr,
                                            curva, importancias)
    res["segundos"] = round(time.time() - t_ini, 1)
    _log(f"  artefactos escritos en {os.path.relpath(ARTEFACTOS, RAIZ)}/{tarea}/ "
         f"({res['segundos']} s)")
    return res


RESULTADOS = {}
for _tarea in TAREAS:
    RESULTADOS[_tarea] = ejecutar_tarea(_tarea)

print("\n" + "=" * 78)
print(f"{'tarea':20s} {'algoritmo':22s} {'selector':9s} {'balanceo':9s} {'s':>6s}")
for _tarea, _r in RESULTADOS.items():
    print(f"{_tarea:20s} {_r['config']['algoritmo']:22s} {_r['config']['selector']:9s} "
          f"{_r['config']['balanceo']:9s} {_r['segundos']:6.0f}")

# Manifiesto de las tres tareas (no pisa el del bloque de modelos)
_manifiesto = {
    "proyecto": "ERCE 2019 · Cuba – 3er grado – Módulo Nacional de Matemática",
    "generado": pd.Timestamp.now().isoformat(),
    "origen_datos": "exploracion.ipynb → modelos/datos/{reg,clf}_indice_*.csv",
    "proceso": ("Weka-fiel: hold-out 20 % + CV 5 (4 AttributeSelection) + "
                "balanceo por IR + Friedman/Nemenyi + CV 10 + estabilidad 3×5"),
    "cribado": {"algoritmos": {t: EXT.grupos(TAREAS[t]["tipo"])[0]
                               + EXT.grupos(TAREAS[t]["tipo"])[1]
                               for t in TAREAS},
                "afinado": {"iteraciones": EXT.N_AFINADO,
                            "folds": EXT.CV_AFINADO, "top": N_AFINADO_TOP},
                "contraste_linea_base": "t pareado + Wilcoxon + corrección de Holm"},
    "tareas": {t: {"tipo": r["artefacto"]["tipo"], "descripcion": r["artefacto"]["descripcion"],
                   "target": r["artefacto"]["target"], "clases": r["artefacto"]["clases"],
                   "algoritmo": r["config"]["algoritmo"], "selector": r["config"]["selector"],
                   "selector_weka": r["artefacto"]["selector_weka"],
                   "balanceo": r["config"]["balanceo"],
                   "estimacion": r["artefacto"]["estimacion"],
                   "friedman": r["artefacto"]["friedman"]}
               for t, r in RESULTADOS.items()},
}
with open(os.path.join(ARTEFACTOS, "manifiesto.json"), "w", encoding="utf-8") as f:
    json.dump(_manifiesto, f, ensure_ascii=False, indent=2, default=str)
print(f"\nmanifiesto: {os.path.relpath(os.path.join(ARTEFACTOS, 'manifiesto.json'), RAIZ)}")

## 19. Lectura de resultados

Tres tablas resumen las tres tareas: **número contra línea base**, **estabilidad**
y **brecha**.

In [ ]:
resumen_modelos = pd.DataFrame([{
    "tarea": t,
    "tipo": r["artefacto"]["tipo"],
    "algoritmo": r["config"]["algoritmo"],
    "selector Weka": ETIQUETAS_WEEKA[r["config"]["selector"]],
    "balanceo": r["config"]["balanceo"],
    "holdout (métrica)": r["holdout"][r["metrica_principal"]],
    "línea base": r["linea_base_holdout"][r["metrica_principal"]],
    "supera la base": r["linea_base_holdout"]["supera_a_linea_base"] > 0,
    f"CV {mcfg.CV_FOLDS_FINAL}": r["cv_final"][f"{principal_de(TAREAS[t]['tipo'])}_media"],
    "estabilidad 3×5": r["estabilidad"]["media"],
    "brecha": r["brecha"]["estado"],
    "Friedman p": r["friedman"]["friedman_p"],
    "variables usadas": len(r["nombres_sel"]),
} for t, r in RESULTADOS.items()])

print("detalle de la línea base de cada tarea:")
for t, r in RESULTADOS.items():
    print(f"  {t:20s} hold-out {r['holdout']} ")
    print(f"  {'':20s} base    { {k: v for k, v in r['linea_base_holdout'].items()} }")

resumen_modelos

In [ ]:
# Gráfico 1: ¿cada modelo supera su línea base?
fig = go.Figure()
etiquetas, modelos_v, base_v, colores = [], [], [], []
for t, r in RESULTADOS.items():
    et = r["metrica_principal"]
    v, b = r["holdout"][et], r["linea_base_holdout"][et]
    modelos_v.append(v)
    base_v.append(b)
    colores.append("#1c7c4d" if r["linea_base_holdout"]["supera_a_linea_base"] > 0 else "#c0392b")
    etiquetas.append(f"{t}<br>{et}")

for i, t in enumerate(RESULTADOS):
    fig.add_bar(x=[etiquetas[i]], y=[base_v[i]], name="línea base", offsetgroup=0,
                marker_color="#b9c0cc", showlegend=(i == 0),
                hovertemplate=f"{t}<br>línea base: %{{y:.3f}}<extra></extra>")
    fig.add_bar(x=[etiquetas[i]], y=[modelos_v[i]], name="modelo", offsetgroup=1,
                marker_color=colores[i], showlegend=(i == 0),
                hovertemplate=f"{t}<br>modelo: %{{y:.3f}}<extra></extra>")
    fig.add_annotation(x=etiquetas[i], y=modelos_v[i], text=f"{modelos_v[i]:.3f}",
                       showarrow=False, yshift=10, font={"size": 12})

fig.update_layout(
    title={"text": "Modelo vs línea base en el hold-out reservado "
                   "(verde = supera la base, rojo = no la supera)",
           "x": 0.5, "xanchor": "center", "font": {"size": 16}},
    barmode="group", height=430, template="plotly_white",
    margin={"l": 60, "r": 20, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12},
    yaxis={"title": "métrica principal de cada tarea", "gridcolor": "#e8e8e8"})
fig.show()

In [ ]:
# Gráfico 2: ranking de Friedman (rango 1 = mejor) y diferencia crítica
fig = go.Figure()
for i, (t, r) in enumerate(RESULTADOS.items()):
    rank = r["friedman"]["ranking"].sort_values("rango_medio", ascending=False)
    fig.add_trace(go.Scatter(
        y=rank.algoritmo + f"  ({t})", x=rank.rango_medio, mode="markers+text",
        text=[f"{v:.2f}" for v in rank.rango_medio], textposition="middle right",
        marker={"size": 11, "color": ["#1f3a93", "#4aa3df", "#8c8c8c"][i]},
        name=t, showlegend=False,
        error_x={"type": "data", "array": [r["friedman"]["cd_nemenyi"] / 2] * len(rank),
                 "visible": True, "color": "#9e9e9e", "thickness": 1.5, "width": 4}))

fig.update_layout(
    title={"text": "Rango medio por algoritmo (Friedman, bloques = folds) · "
                   "las barras son ± CD/2 de Nemenyi",
           "x": 0.5, "xanchor": "center", "font": {"size": 16}},
    height=620, template="plotly_white", margin={"l": 260, "r": 60, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 11},
    xaxis={"title": "rango medio (1 = mejor)", "dtick": 1,
           "gridcolor": "#e8e8e8", "autorange": "reversed"},
    yaxis={"automargin": True})
fig.show()

In [ ]:
# Gráfico 3: matriz de confusión / dispersión según el tipo de tarea
fig = go.Figure()
tiles = []
for t, r in RESULTADOS.items():
    if TAREAS[t]["tipo"] != "clasificacion":
        continue
    clases = r["detalle"]["clases"]
    cm = np.asarray(r["detalle"]["matriz"], dtype=float)
    norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    fig.add_trace(go.Heatmap(
        z=norm, x=[f"pred {c}" for c in clases], y=[f"real {c}" for c in clases],
        colorscale=[[0, "#fbfbfa"], [1, "#1f3a93"]], zmin=0, zmax=1,
        text=cm.astype(int), texttemplate="%{text}", colorbar={"title": "% fila"},
        name=t, showscale=(t == "clf_indice_bajo")))

fig.update_layout(
    title={"text": "Matrices de confusión en el hold-out (número y % de la fila real)",
           "x": 0.5, "xanchor": "center", "font": {"size": 16}},
    height=420, template="plotly_white", margin={"l": 70, "r": 20, "t": 60, "b": 60},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12})
fig.show()

_reg = RESULTADOS.get("reg_indice_global")
if _reg is not None:
    _fig = go.Figure()
    _obs = np.asarray(_reg["yte"], dtype=float)
    _pred = np.asarray(_reg["y_te_pred"], dtype=float)
    _fig.add_scatter(x=_obs, y=_pred, mode="markers",
                     marker={"size": 7, "opacity": 0.35, "color": "#4aa3df"},
                     name="estudiantes", showlegend=False,
                     hovertemplate="observado %{x:.0f}<br>predicho %{y:.1f}<extra></extra>")
    _lim = [min(_obs.min(), _pred.min()), max(_obs.max(), _pred.max())]
    _fig.add_scatter(x=_lim, y=_lim, mode="lines", line={"dash": "dash", "color": "#c0392b"},
                     name="y = x")
    _fig.update_layout(
        title={"text": f"reg_indice_global · observado vs predicho "
                       f"(R² = {_reg['holdout']['r2']:.3f}, "
                       f"línea base R² = {_reg['linea_base_holdout']['r2']:.3f})",
               "x": 0.5, "xanchor": "center", "font": {"size": 16}},
        height=460, template="plotly_white", margin={"l": 60, "r": 20, "t": 60, "b": 50},
        font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12},
        xaxis={"title": "Indice_global observado", "range": _lim, "gridcolor": "#e8e8e8"},
        yaxis={"title": "predicho", "range": _lim, "gridcolor": "#e8e8e8"})
    _fig.show()

In [ ]:
# Gráfico 4: curva de aprendizaje del ganador de cada tarea
fig = go.Figure()
for i, (t, r) in enumerate(RESULTADOS.items()):
    curva = r["curva"]
    color = ["#1f3a93", "#4aa3df", "#8c8c8c"][i]
    fig.add_scatter(x=[c["n_entrenamiento"] for c in curva],
                    y=[c["test_media"] for c in curva], mode="lines+markers",
                    line={"color": color}, marker={"size": 7},
                    error_y={"type": "data",
                             "array": [c["test_de"] for c in curva], "visible": True,
                             "color": color, "thickness": 1.2},
                    name=f"{t} · validación")
    fig.add_scatter(x=[c["n_entrenamiento"] for c in curva],
                    y=[c["train_media"] for c in curva], mode="lines+markers",
                    line={"color": color, "dash": "dot"}, marker={"size": 6},
                    name=f"{t} · entrenamiento")

fig.update_layout(
    title={"text": "Curvas de aprendizaje (cadena completa, 5 folds) · "
                   "el hueco train–validación es el sobreajuste",
           "x": 0.5, "xanchor": "center", "font": {"size": 16}},
    height=470, template="plotly_white", margin={"l": 60, "r": 20, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12},
    xaxis={"title": "filas de entrenamiento", "gridcolor": "#e8e8e8"},
    yaxis={"title": "puntaje de la CV interna", "gridcolor": "#e8e8e8"})
fig.show()

In [ ]:
# Gráfico 5: importancia de variables del ganador de cada tarea
fig = go.Figure()
for i, (t, r) in enumerate(RESULTADOS.items()):
    top = r["importancias"].head(8).iloc[::-1]
    fig.add_trace(go.Bar(
        y=top.variable, x=top.importancia, orientation="h",
        marker_color=["#1f3a93", "#4aa3df", "#8c8c8c"][i],
        name=f"{t} ({r['metodo_importancia'].split(' ')[0]})",
        hovertemplate=t + "<br>%{y}: %{x:.4f}<extra></extra>"))

fig.update_layout(
    title={"text": "Importancia de variables del modelo ganador (hold-out)",
           "x": 0.5, "xanchor": "center", "font": {"size": 16}},
    barmode="group", height=max(420, 26 * 8), template="plotly_white",
    margin={"l": 210, "r": 20, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 11},
    xaxis={"title": "importancia", "gridcolor": "#e8e8e8"},
    yaxis={"automargin": True}, legend={"title": "tarea"})
fig.show()

pd.concat([r["importancias"].head(5).assign(tarea=t) for t, r in RESULTADOS.items()],
          ignore_index=True)

## 20. Qué se puede afirmar

Las conclusiones se calculan en la celda siguiente a partir de los resultados, en
lugar de escribirlas a mano: así el cuaderno no puede quedarse diciendo algo que
los datos ya no sostienen.

In [ ]:
def veredicto(t, r):
    et = r["metrica_principal"]
    mejor_cv = r["comparativa"].iloc[0]
    dif = r["linea_base_holdout"]["supera_a_linea_base"]
    if dif > 0:
        return (f"**{t}**: el ganador ({r['config']['algoritmo']}) mejora la línea base "
                f"en {dif:.4f} de {et} en el hold-out. Conviene mirar si el CV 10 "
                f"({r['cv_final'][f'{principal_de(TAREAS[t]['tipo'])}_media']:.4f}) "
                f"y la estabilidad ({r['estabilidad']['media']:.4f} ± "
                f"{r['estabilidad']['de']:.4f}) sostienen esa diferencia.")
    return (f"**{t}**: el ganador ({r['config']['algoritmo']}, "
            f"{ETIQUETAS_WEEKA[r['config']['selector']]}) queda en "
            f"{r['holdout'][et]:.4f} frente a una línea base de "
            f"{r['linea_base_holdout'][et]:.4f} ({dif:+.4f}). No hay modelo que "
            f"supere predecir siempre lo más frecuente; el mejor resultado de la CV "
            f"de selección fue {mejor_cv['principal_mean']:.4f} "
            f"({mejor_cv['algoritmo']}), tampoco sobre la base. Estado del "
            f"diagnóstico: **{r['brecha']['estado']}**.")


print("Veredictos por tarea\n" + "-" * 78)
for t, r in RESULTADOS.items():
    print(veredicto(t, r) + "\n")

print("Friedman")
for t, r in RESULTADOS.items():
    _f = r["friedman"]
    print(f"  {t:20s} p={_f['friedman_p']:.4f} "
          f"{'significativo' if _f['significativo'] else 'NO significativo'} · "
          f"CD={_f['cd_nemenyi']:.2f} · mejor rango: "
          f"{_f['ranking'].iloc[0]['algoritmo']} ({_f['ranking'].iloc[0]['rango_medio']:.2f})")

print("\nSobreajuste y muestra")
for t, r in RESULTADOS.items():
    print(f"  {t:20s} brecha={r['brecha']['brecha']:+.4f} ({r['brecha']['estado']}) · "
          f"validación {r['curva'][0]['test_media']:.3f} → {r['curva'][-1]['test_media']:.3f} "
          f"con más datos · {len(r['nombres_sel'])} variables en el modelo final")

print("\nSobre el instrumento, no sobre el modelo")
print("  Las variables disponibles no se asocian con el objetivo (|r| ≤ 0,06 en "
      "cuadernos/exploracion.ipynb §7):")
print("  con estas variables no hay nada que aprender. El diseño correcto para "
      "estimar el desempeño del 3er grado es unir el puntaje de mathematics "
      "(PM3.csv), como ya hace el bloque de modelos.")

## 21. Contrato de entrada y mapa de archivos

`metadata.json` fija qué columnas espera el pipeline de despliegue. Se comprueba
con `modelos/contrato.validar_y_tipar`, el mismo punto de entrada que usa el
software de escritorio.

In [ ]:
for t, r in RESULTADOS.items():
    meta = r["artefacto"]
    df = DATAFRAMES[t]
    muestra = df[meta["features_esperadas"]].head(50)
    chk = CTR.validar_y_tipar(muestra, meta)
    # Y el pipeline de despliegue predice sin romperse:
    cargados = joblib.load(os.path.join(ARTEFACTOS, t, "pipeline.joblib"))
    pred = cargados["pipeline"].named_steps["mod"].predict(
        cargados["pipeline"].named_steps["sel"].transform(
            cargados["pipeline"].named_steps["pre"].transform(chk["df"])))
    print(f"{t:20s} contrato ok={chk['ok']} · faltantes={chk['faltantes']} · "
          f"sobrantes={len(chk['sobrantes'])} · predice {len(pred)} filas de prueba")

In [ ]:
# Mapa final: qué se escribió y qué hay que regenerar
inventario = []
for carpeta, _, archivos in os.walk(ARTEFACTOS):
    for a in sorted(archivos):
        ruta = os.path.join(carpeta, a)
        inventario.append({"ruta": os.path.relpath(ruta, RAIZ),
                           "KB": f"{os.path.getsize(ruta) / 1024:.1f}"})
pd.DataFrame(inventario)

---

## Cómo se reproduce

```bash
entorno/bin/python -m jupyter nbconvert --to notebook --execute --inplace cuadernos/exploracion.ipynb
entorno/bin/python -m jupyter nbconvert --to notebook --execute --inplace cuadernos/estimacion.ipynb
```

`cuadernos/exploracion.ipynb` debe correr primero: escribe los tres datasets que este
cuaderno carga.

### Dónde está cada cosa

| Ruta | Qué es |
|---|---|
| `cuadernos/exploracion.ipynb` | EDA, curación con bitácora y selección de características |
| `cuadernos/estimacion.ipynb` | este cuaderno: modelos y estimación |
| `modelos/datos/{reg,clf}_indice_*.csv` | datasets curados (regenerables, ignorados por git) |
| `modelos/datos/seleccion_indice_global.csv` | frecuencia de selección de las 28 variables |
| `artefactos/indice_global/<tarea>/` | pipeline de despliegue, metadata, métricas, importancias |
| `artefactos/indice_global/reporte/<tarea>/` | tablas por fold y figuras (PNG/SVG/PDF) |
| `artefactos/indice_global/manifiesto.json` | resumen de las tres tareas |

Los artefactos de los cinco modelos del bloque de modelos
(`artefactos/clf_*`, `artefactos/reg_mat`, `artefactos/manifiesto.json`) no se
tocan.